In [6]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
import soundfile as sf
from tqdm import tqdm
import glob

class FBankExtractor:
    def __init__(self, device, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160):
        self.sample_rate = sample_rate
        self.device = device
        # Khởi tạo transform MelSpectrogram trực tiếp trên GPU
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, 
            n_fft=n_fft, 
            n_mels=n_mels, 
            hop_length=hop_length, 
            center=False
        ).to(self.device)

    def _apply_cmvn(self, feature):
        """Chuẩn hóa Mean and Variance Normalization"""
        mean = feature.mean(dim=-1, keepdim=True)
        std = feature.std(dim=-1, keepdim=True)
        return (feature - mean) / (std + 1e-6)

    def extract(self, waveform_cpu):
        # Đưa dữ liệu lên GPU
        waveform_gpu = waveform_cpu.to(self.device)
        if waveform_gpu.dim() == 1: 
            waveform_gpu = waveform_gpu.unsqueeze(0)

        # Tính toán FBank (Log-Mel Spectrogram)
        with torch.no_grad():
            mel_spec = self.mel_transform(waveform_gpu)
            fbank = torch.log(mel_spec + 1e-6)
            fbank = self._apply_cmvn(fbank)
        
        return fbank.squeeze(0).cpu() # Đẩy về CPU để tránh tràn VRAM khi gom dict

# --- 1. CẤU HÌNH ĐƯỜNG DẪN ---
AUDIO_INPUT_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\test-O"
OUTPUT_FILE_PT = r"D:\Study\7-SP26\DATxSLP\Test set O\feature_dat\test_O_fbank.pt"
SAMPLE_RATE = 16000

# --- 2. KHỞI TẠO ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Đang chạy trên: {device}")

extractor = FBankExtractor(device=device, sample_rate=SAMPLE_RATE)
audio_files = sorted(glob.glob(os.path.join(AUDIO_INPUT_DIR, "**", "*.wav"), recursive=True) + \
                     glob.glob(os.path.join(AUDIO_INPUT_DIR, "**", "*.flac"), recursive=True))

print(f"🔍 Tìm thấy {len(audio_files)} file âm thanh.")

# SỬA Ở ĐÂY: Dùng 1 dictionary duy nhất thay vì 2 list riêng biệt
features_dict = {}

# --- 3. TIẾN TRÌNH TRÍCH XUẤT ---
for path in tqdm(audio_files, desc="Đang trích xuất & Gom dữ liệu"):
    try:
        # Đọc file âm thanh
        wav, sr = sf.read(path)
        waveform = torch.from_numpy(wav).float()
        
        # Chuẩn hóa mono & Resample nếu cần
        if waveform.dim() > 1:
            # Nếu audio là stereo, lấy trung bình cộng các kênh
            waveform = torch.mean(waveform, dim=1 if wav.shape[0] > wav.shape[1] else 0, keepdim=True)
        else:
            waveform = waveform.unsqueeze(0)
            
        if sr != SAMPLE_RATE:
            resampler = T.Resample(sr, SAMPLE_RATE)
            waveform = resampler(waveform)

        # Trích xuất đặc trưng
        fbank = extractor.extract(waveform)
        
        # SỬA Ở ĐÂY: Gán trực tiếp tensor vào dictionary với key là tên file
        filename = os.path.basename(path)
        features_dict[filename] = fbank.half()
        
    except Exception as e:
        print(f"⚠ Lỗi xử lý {path}: {e}")

# --- 4. LƯU FILE DUY NHẤT ---
if features_dict:
    print(f"💾 Đang lưu {len(features_dict)} mẫu vào: {OUTPUT_FILE_PT}...")
    os.makedirs(os.path.dirname(OUTPUT_FILE_PT), exist_ok=True)
    
    # SỬA Ở ĐÂY: Lưu trực tiếp features_dict
    torch.save(features_dict, OUTPUT_FILE_PT)
    print("✅ Đã hoàn thành! Khi load file .pt này lên, hàm len() sẽ trả về đúng số lượng file (vd: 17786).")
else:
    print("❌ Không có dữ liệu để lưu. Vui lòng kiểm tra lại thư mục đầu vào.")

🔥 Đang chạy trên: cuda
🔍 Tìm thấy 17786 file âm thanh.


Đang trích xuất & Gom dữ liệu: 100%|██████████| 17786/17786 [00:38<00:00, 458.61it/s]


💾 Đang lưu 17786 mẫu vào: D:\Study\7-SP26\DATxSLP\Test set O\feature_dat\test_O_fbank.pt...
✅ Đã hoàn thành! Khi load file .pt này lên, hàm len() sẽ trả về đúng số lượng file (vd: 17786).
